# Ética y Buenas Prácticas en Inteligencia Artificial

**Cloud, IA y Data Science — Ciclo 1**

---

En las sesiones anteriores hemos aprendido a **construir modelos de IA**: clasificadores, árboles de decisión, generación de datos sintéticos… Hemos visto que un modelo puede alcanzar buenas métricas de rendimiento. Pero hay una pregunta que todavía no nos hemos hecho:

> **¿Que un modelo sea preciso significa que sea justo?**

Esta sesión trata exactamente de eso. Vamos a explorar qué puede salir mal cuando la IA se despliega en el mundo real y qué responsabilidad tenemos como profesionales de datos.

## 1. Los cinco pilares de la IA ética

Cuando hablamos de ética en IA, no estamos hablando de filosofía abstracta. Estamos hablando de **decisiones técnicas con impacto real en personas reales**. Los cinco pilares fundamentales son:

### 1.1 Sesgo (Bias)

Un modelo aprende de datos históricos. Si esos datos reflejan discriminación pasada, el modelo la **reproduce y amplifica**.

**Caso real — Amazon (2018):** Amazon desarrolló un sistema de IA para filtrar currículos de candidatos. El modelo se entrenó con datos de contrataciones de los últimos 10 años. Como la mayoría de contratados habían sido hombres, el modelo aprendió a **penalizar currículos que contenían la palabra "women's"** (por ejemplo, "women's chess club"). Amazon tuvo que descartar el sistema.

### 1.2 Equidad (Fairness)

¿Trata el modelo de forma equivalente a diferentes grupos de personas? No basta con que el accuracy global sea alto; hay que verificar que **no hay grupos sistemáticamente perjudicados**.

**Caso real — COMPAS:** Un algoritmo usado en tribunales de EE.UU. para predecir reincidencia criminal. Investigaciones de ProPublica revelaron que el sistema asignaba puntuaciones de riesgo significativamente más altas a personas afroamericanas que a personas blancas, incluso cuando el historial delictivo era comparable.

### 1.3 Transparencia (Explainability)

¿Podemos entender **por qué** el modelo toma una decisión? Un modelo de caja negra que deniega un crédito sin explicación es un problema tanto ético como legal (en la UE, el RGPD reconoce el derecho a una explicación).

### 1.4 Privacidad

Los modelos de IA necesitan datos, a menudo datos personales. La privacidad exige:
- Recoger **solo los datos necesarios** (minimización).
- Protegerlos adecuadamente.
- Que el usuario sepa qué se hace con sus datos y pueda **revocar el consentimiento**.

**Caso real — Clearview AI:** Una empresa que creó una base de datos de reconocimiento facial con miles de millones de fotos extraídas de redes sociales sin consentimiento. Ha sido sancionada en múltiples países europeos.

### 1.5 Responsabilidad (Accountability)

Cuando un sistema de IA causa daño, **¿quién responde?** ¿El desarrollador? ¿La empresa? ¿El usuario? La IA ética exige que haya una cadena clara de responsabilidad y que existan mecanismos de auditoría y corrección.

---

## 2. Caso práctico: Detectando sesgo en un modelo de crédito

Vamos a construir un escenario realista. Imaginemos que trabajamos en un banco y nos piden crear un modelo para **aprobar o denegar solicitudes de crédito**. Vamos a simular datos, entrenar un modelo y luego **auditar si es justo**.

### 2.1 Generar datos sintéticos

Generamos un dataset con un sesgo intencionado: las personas del Grupo B tienen ingresos ligeramente más bajos en promedio (reflejo de desigualdades históricas). Esto hará que el modelo, sin ser explícitamente discriminatorio, **aprenda ese sesgo**.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, confusion_matrix

np.random.seed(42)

n = 1000

# Atributo protegido: grupo A y grupo B
grupo = np.random.choice(['A', 'B'], size=n, p=[0.5, 0.5])

# Ingresos: el grupo B tiene ingresos algo más bajos en promedio (sesgo histórico)
ingresos = np.where(
    grupo == 'A',
    np.random.normal(45000, 12000, n),   # Grupo A: media 45k
    np.random.normal(38000, 12000, n)    # Grupo B: media 38k (brecha salarial histórica)
)
ingresos = np.clip(ingresos, 10000, 100000)  # Limitar a rango realista

# Antigüedad laboral (años)
antiguedad = np.random.exponential(5, n).clip(0, 30).round(1)

# Deuda actual (proporción sobre ingresos)
ratio_deuda = np.random.beta(2, 5, n).round(3)

# Variable objetivo: aprobación del crédito
# La regla real depende de ingresos, antigüedad y deuda
# Pero como los ingresos están sesgados, el resultado también lo estará
score = (
    0.4 * (ingresos / 100000) +
    0.3 * (antiguedad / 30) +
    0.3 * (1 - ratio_deuda)
)
probabilidad = 1 / (1 + np.exp(-10 * (score - 0.35)))
aprobado = np.random.binomial(1, probabilidad)

# Crear DataFrame
df = pd.DataFrame({
    'grupo': grupo,
    'ingresos': ingresos.round(0).astype(int),
    'antiguedad_laboral': antiguedad,
    'ratio_deuda': ratio_deuda,
    'credito_aprobado': aprobado
})

print(f"Dataset: {df.shape[0]} solicitudes, {df.shape[1]} columnas")
print(f"\nTasa de aprobación global: {df['credito_aprobado'].mean():.1%}")
df.head(10)

### 2.2 Primera señal de alerta: diferencias entre grupos

Antes de entrenar ningún modelo, un analista de datos responsable debería mirar cómo se distribuyen las variables entre grupos. Esto es parte del **EDA ético**: no solo exploramos los datos para modelar mejor, sino para detectar **posibles fuentes de sesgo**.

In [ ]:
# Tasa de aprobación por grupo en los datos originales
tasa_por_grupo = df.groupby('grupo')['credito_aprobado'].mean()
print("=" * 40)
print("TASA DE APROBACIÓN POR GRUPO (datos reales)")
print("=" * 40)
for g, tasa in tasa_por_grupo.items():
    print(f"  Grupo {g}: {tasa:.1%}")
print(f"\n  Diferencia: {abs(tasa_por_grupo['A'] - tasa_por_grupo['B']):.1%}")
print("\nEsta diferencia ya existe en los datos históricos.")
print("Si el modelo aprende de estos datos, reproducirá esta desigualdad.")

In [ ]:
# Visualizar la distribución de ingresos por grupo
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Ingresos por grupo
for g, color in zip(['A', 'B'], ['steelblue', 'coral']):
    subset = df[df['grupo'] == g]
    axes[0].hist(subset['ingresos'], bins=30, alpha=0.6, label=f'Grupo {g}', color=color)
axes[0].set_xlabel('Ingresos anuales (€)')
axes[0].set_ylabel('Frecuencia')
axes[0].set_title('Distribución de ingresos por grupo')
axes[0].legend()

# Tasa de aprobación por grupo
tasas = df.groupby('grupo')['credito_aprobado'].mean()
bars = axes[1].bar(tasas.index, tasas.values, color=['steelblue', 'coral'], edgecolor='black')
axes[1].set_ylabel('Tasa de aprobación')
axes[1].set_title('Tasa de aprobación por grupo (datos históricos)')
axes[1].set_ylim(0, 1)
for bar, val in zip(bars, tasas.values):
    axes[1].text(bar.get_x() + bar.get_width()/2, val + 0.02, f'{val:.1%}',
                ha='center', fontweight='bold')

plt.tight_layout()
plt.show()

### 2.3 Entrenar el modelo

Ahora hacemos lo que normalmente haríamos: entrenar un clasificador para predecir si un crédito se aprueba o no. Usaremos un **árbol de decisión** (que ya conocemos de sesiones anteriores).

**Pregunta clave:** ¿Incluimos la variable `grupo` como feature? Muchas empresas piensan que con **eliminar el atributo protegido** ya eliminan el sesgo. Vamos a comprobar si eso es cierto.

In [ ]:
# Entrenar SIN incluir el grupo como feature
# (intento de ser "justo" eliminando el atributo protegido)
X = df[['ingresos', 'antiguedad_laboral', 'ratio_deuda']]
y = df['credito_aprobado']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

modelo = DecisionTreeClassifier(max_depth=4, random_state=42)
modelo.fit(X_train, y_train)

y_pred = modelo.predict(X_test)

print(f"Accuracy global del modelo: {accuracy_score(y_test, y_pred):.1%}")
print(f"\n¡Buen rendimiento! Pero... ¿es justo?")

### 2.4 Auditoría de equidad

Aquí está el paso que muchos equipos de datos **se saltan**. No basta con mirar el accuracy global. Necesitamos desglosar las predicciones por grupo y ver si el modelo trata a ambos de forma comparable.

Vamos a medir dos cosas:
- **Tasa de aprobación predicha por grupo**: ¿El modelo aprueba créditos a la misma tasa para ambos grupos?
- **Accuracy por grupo**: ¿El modelo acierta igual de bien para ambos?

In [ ]:
# Recuperar el grupo para las muestras de test
grupos_test = df.loc[X_test.index, 'grupo']

# Análisis por grupo
print("=" * 55)
print("AUDITORÍA DE EQUIDAD DEL MODELO")
print("=" * 55)

for g in ['A', 'B']:
    mask = grupos_test == g
    n_grupo = mask.sum()
    acc_grupo = accuracy_score(y_test[mask], y_pred[mask])
    tasa_aprobacion = y_pred[mask].mean()
    tasa_real = y_test[mask].mean()
    
    print(f"\n  Grupo {g} ({n_grupo} muestras):")
    print(f"    Accuracy:                {acc_grupo:.1%}")
    print(f"    Tasa aprobación predicha: {tasa_aprobacion:.1%}")
    print(f"    Tasa aprobación real:     {tasa_real:.1%}")

# Disparidad
tasa_a = y_pred[grupos_test == 'A'].mean()
tasa_b = y_pred[grupos_test == 'B'].mean()
disparidad = tasa_a - tasa_b

print(f"\n{'=' * 55}")
print(f"  DISPARIDAD en tasa de aprobación: {disparidad:.1%}")
print(f"{'=' * 55}")

if abs(disparidad) > 0.05:
    print("\n  ⚠️  El modelo muestra una diferencia significativa entre grupos.")
    print("  Aunque NO usamos 'grupo' como variable, el sesgo se filtra")
    print("  a través de los ingresos (que están correlacionados con el grupo).")
    print("\n  Esto se llama SESGO INDIRECTO o PROXY BIAS.")
else:
    print("\n  ✅  La disparidad es pequeña.")

In [ ]:
# Visualizar la disparidad
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Tasa de aprobación predicha vs real por grupo
grupos = ['A', 'B']
tasas_pred = [y_pred[grupos_test == g].mean() for g in grupos]
tasas_real = [y_test[grupos_test == g].mean() for g in grupos]

x_pos = np.arange(len(grupos))
width = 0.35

bars1 = axes[0].bar(x_pos - width/2, tasas_real, width, label='Real (datos)', 
                     color='lightgray', edgecolor='black')
bars2 = axes[0].bar(x_pos + width/2, tasas_pred, width, label='Predicha (modelo)',
                     color=['steelblue', 'coral'], edgecolor='black')
axes[0].set_xticks(x_pos)
axes[0].set_xticklabels(['Grupo A', 'Grupo B'])
axes[0].set_ylabel('Tasa de aprobación')
axes[0].set_title('Aprobación real vs. predicha por grupo')
axes[0].legend()
axes[0].set_ylim(0, 1)

# Accuracy por grupo
accs = [accuracy_score(y_test[grupos_test == g], y_pred[grupos_test == g]) for g in grupos]
bars3 = axes[1].bar(grupos, accs, color=['steelblue', 'coral'], edgecolor='black')
axes[1].set_ylabel('Accuracy')
axes[1].set_title('Accuracy del modelo por grupo')
axes[1].set_ylim(0, 1)
for bar, val in zip(bars3, accs):
    axes[1].text(bar.get_x() + bar.get_width()/2, val + 0.02, f'{val:.1%}',
                ha='center', fontweight='bold')

plt.tight_layout()
plt.show()

### 2.5 Lección clave: el proxy bias

Acabamos de ver algo muy importante: **eliminar el atributo protegido no elimina el sesgo**.

¿Por qué? Porque hay otras variables en el dataset que están **correlacionadas** con el grupo. En nuestro ejemplo, los ingresos tienen una distribución diferente para cada grupo (reflejo de desigualdades históricas). El modelo usa los ingresos como proxy del grupo sin necesidad de verlo directamente.

Esto se llama **proxy bias** o **sesgo indirecto**, y es uno de los problemas más difíciles de resolver en IA ética. La solución no es simplemente "quitar la columna".

---

## 3. Buenas prácticas y estrategias de mitigación

Ahora que hemos visto el problema, ¿qué podemos hacer? Hay tres momentos clave para intervenir:

### Antes del entrenamiento (Pre-procesamiento)
- **Auditar los datos** buscando desbalances entre grupos protegidos.
- **Rebalancear** los datos si un grupo está infrarrepresentado.
- **Cuestionar los datos históricos**: si los datos reflejan discriminación pasada, quizás no deberían ser la única base del modelo.

### Durante el entrenamiento (In-procesamiento)
- Utilizar **restricciones de equidad** en el algoritmo (ej. igualar tasas de aprobación entre grupos).
- Probar **modelos más interpretables** (árboles simples, regresión logística) que permitan entender las decisiones.

### Después del entrenamiento (Post-procesamiento)
- **Auditar las predicciones** desglosadas por grupo (exactamente lo que acabamos de hacer).
- Ajustar los umbrales de decisión por grupo si es necesario.
- Implementar **monitorización continua**: el sesgo puede aparecer o empeorar con el tiempo.

### Checklist de IA responsable

Antes de desplegar un modelo en producción, todo equipo debería responder:

| Pregunta | Área |
|----------|------|
| ¿Hemos identificado los grupos que podrían verse afectados? | Equidad |
| ¿Hemos medido métricas de rendimiento por grupo? | Sesgo |
| ¿Puede un humano entender por qué el modelo toma cada decisión? | Transparencia |
| ¿Estamos recogiendo solo los datos necesarios? | Privacidad |
| ¿Hay un proceso para reportar y corregir errores del modelo? | Responsabilidad |
| ¿Existe supervisión humana en las decisiones críticas? | Responsabilidad |

---

## 4. Dilemas éticos para reflexionar

Para cerrar la sesión, vamos a debatir estos escenarios. **No hay respuestas correctas**; el objetivo es desarrollar el pensamiento crítico.

### Dilema 1 — IA en medicina
Un hospital usa IA para priorizar pacientes en urgencias. El modelo es más preciso que los médicos de triaje en el 90% de los casos. Sin embargo, un estudio revela que el modelo tiende a subestimar la gravedad de síntomas en mujeres (porque los datos de entrenamiento reflejan un sesgo histórico en diagnóstico).

**¿Se debería seguir usando el modelo? ¿Bajo qué condiciones?**

---

### Dilema 2 — Vigilancia y seguridad
Una ciudad quiere instalar cámaras con reconocimiento facial en zonas de alta criminalidad. El sistema reduciría el tiempo de identificación de sospechosos de días a segundos.

**¿Dónde está el límite entre seguridad y privacidad? ¿Quién decide?**

---

### Dilema 3 — IA en contratación
Una empresa de tecnología usa IA para filtrar los primeros 10.000 CVs y reducirlos a 500. El sistema es mucho más rápido que los reclutadores humanos, pero no puede evaluar "soft skills" ni contexto personal.

**¿Es aceptable que una IA decida quién pasa a la siguiente fase? ¿Qué se pierde?**

---

### Dilema 4 — Arte generativo y propiedad intelectual
Un diseñador usa una IA generativa (como DALL-E o Midjourney) para crear ilustraciones que vende a clientes. La IA fue entrenada con millones de obras de artistas que no dieron su consentimiento.

**¿De quién es la obra resultante? ¿Se debería compensar a los artistas originales?**

---

## 5. Resumen y cierre

### Lo que hemos aprendido hoy

1. **La precisión no garantiza la equidad.** Un modelo con buen accuracy puede ser profundamente injusto para ciertos grupos.

2. **Eliminar el atributo protegido no basta.** El sesgo se filtra a través de variables proxy. Hay que auditar activamente.

3. **La ética en IA no es opcional.** En Europa, el AI Act establece obligaciones legales para sistemas de IA de alto riesgo (créditos, sanidad, empleo…).

4. **Nuestro papel como analistas de datos.** Nosotros somos los que construimos estos sistemas. Tenemos la responsabilidad de preguntarnos no solo "¿funciona?" sino "¿es justo?".

### Conexión con el resto del curso

Todo lo que hemos visto en este bloque de Cloud, IA y Data Science se conecta:
- Los **datos en la nube** (sesiones 1-3) son la materia prima.
- Los **algoritmos de ML** (sesiones 4-6) son las herramientas.
- El **mini proyecto** (sesión 7) fue la práctica.
- La **ética** (esta sesión) es la brújula.

Sin ética, las herramientas más potentes pueden hacer más daño que bien.

---

## Ejercicio opcional: Investiga un caso real

Busca en internet un caso real de sesgo o fallo ético en IA que no hayamos mencionado en clase. Prepara una breve ficha con:

1. **¿Qué sistema/empresa estaba involucrado?**
2. **¿Qué tipo de sesgo o problema ético se detectó?**
3. **¿Cuál fue la consecuencia?**
4. **¿Cómo se podría haber evitado?**

Algunos términos de búsqueda útiles: "AI bias case study", "algorithmic discrimination", "AI ethics failure", "sesgo algorítmico casos reales".